# MQTT command generator

Programmatically build `mosquitto_pub`/`mosquitto_sub` CLI commands from `device_config.json`, instead of hand-typing the flags each time. Generated commands reference `$MQTT_PASS` (never the literal password) so nothing secret ever lands in a saved notebook cell or its output.

Two ways to use each generated command:
- copy/paste into a terminal that has `MQTT_PASS` exported (see next cell)
- run in-notebook via `run(cmd)`, which substitutes the real password only in-memory at execution time

In [17]:
import json
import os
from pathlib import Path

REPO_ROOT = Path.cwd().parent
config = json.loads((REPO_ROOT / "device_config.json").read_text())

BROKER = config["mqtt_broker"]
PORT = config["mqtt_port"]
USERNAME = config["mqtt_username"]
TOPIC_PUB = config["mqtt_topic_pub"]
TOPIC_SUB = config["mqtt_topic_sub"]
TOPIC_STATUS = config["mqtt_topic_status"]

CA_CERT = REPO_ROOT / "certs" / "ca.pem"
CLIENT_CERT = REPO_ROOT / "certs" / "client.pem"
CLIENT_KEY = REPO_ROOT / "certs" / "private.pem"

# kept only in the process environment - never interpolated into a printed command
os.environ["MQTT_PASS"] = config["mqtt_password"]

print(f"broker={BROKER}:{PORT} username={USERNAME}")
print(f"pub={TOPIC_PUB} sub={TOPIC_SUB} status={TOPIC_STATUS}")

broker=192.168.1.216:8883 username=dev_fa6648eb
pub=devices/dev_fa6648eb/sensors sub=devices/dev_fa6648eb/commands status=devices/dev_fa6648eb/status


## Command builders

In [18]:
import shlex
import subprocess


def build_pub_command(topic, payload, retain=False, qos=None):
    parts = [
        "mosquitto_pub",
        "-h", BROKER,
        "-p", str(PORT),
        "--cafile", str(CA_CERT),
        "--cert", str(CLIENT_CERT),
        "--key", str(CLIENT_KEY),
        "-u", USERNAME,
        "-P", "$MQTT_PASS",
        "-t", topic,
        "-m", payload,
    ]
    if qos is not None:
        parts += ["-q", str(qos)]
    if retain:
        parts.append("-r")
    return " ".join(shlex.quote(p) if p != "$MQTT_PASS" else p for p in parts)


def build_sub_command(topic, qos=None, verbose=True):
    parts = [
        "mosquitto_sub",
        "-h", BROKER,
        "-p", str(PORT),
        "--cafile", str(CA_CERT),
        "--cert", str(CLIENT_CERT),
        "--key", str(CLIENT_KEY),
        "-u", USERNAME,
        "-P", "$MQTT_PASS",
        "-t", topic,
    ]
    if qos is not None:
        parts += ["-q", str(qos)]
    if verbose:
        parts.append("-v")
    return " ".join(shlex.quote(p) if p != "$MQTT_PASS" else p for p in parts)


def run(cmd, timeout=10):
    """Execute a generated command, substituting the real password only for this call."""
    resolved = cmd.replace("$MQTT_PASS", os.environ["MQTT_PASS"])
    result = subprocess.run(
        shlex.split(resolved), capture_output=True, text=True, timeout=timeout
    )
    result.args = cmd  # mask before returning - Jupyter auto-displays a cell's last
    # expression via repr(), and CompletedProcess.args would otherwise echo the
    # resolved (real-password) argv into the saved notebook output
    print("$", cmd)  # masked - password never printed
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    return result

## Generate one command

Adjust `SUFFIX`/`COMMAND`, rerun. `print()` only, doesn't execute - paste into a terminal with `MQTT_PASS` exported, or pass to `run(...)`.

In [19]:
SUFFIX = "relay"
COMMAND = "on"

cmd = build_pub_command(f"{TOPIC_SUB}/{SUFFIX}", COMMAND)
print(cmd)

mosquitto_pub -h 192.168.1.216 -p 8883 --cafile /Users/nerdmonkey/Productions/microweaver-framework/certs/ca.pem --cert /Users/nerdmonkey/Productions/microweaver-framework/certs/client.pem --key /Users/nerdmonkey/Productions/microweaver-framework/certs/private.pem -u dev_fa6648eb -P $MQTT_PASS -t devices/dev_fa6648eb/commands/relay -m on


## Preview the resolved command (screen only)

Copies the fully-substituted command (real password included) to the clipboard instead of printing it, so the resolved value never appears in a cell output that could get saved or committed. Paste straight into a terminal.

In [20]:
import platform
import shutil


def copy_to_clipboard(text):
    system = platform.system()
    if system == "Darwin" and shutil.which("pbcopy"):
        tool = ["pbcopy"]
    elif shutil.which("xclip"):
        tool = ["xclip", "-selection", "clipboard"]
    elif shutil.which("xsel"):
        tool = ["xsel", "--clipboard", "--input"]
    elif system == "Windows" or shutil.which("clip.exe"):
        tool = ["clip.exe"]
    else:
        return False
    subprocess.run(tool, input=text, text=True, check=True)
    return True


def preview(cmd):
    """Resolve $MQTT_PASS and copy to clipboard - never printed, never saved as output."""
    resolved = cmd.replace("$MQTT_PASS", os.environ["MQTT_PASS"])
    if copy_to_clipboard(resolved):
        print(f"resolved command copied to clipboard ({len(resolved)} chars, password included)")
    else:
        print("no clipboard tool found (install xclip or xsel) - falling back to masked print")
        print(cmd)


preview(cmd)

resolved command copied to clipboard (345 chars, password included)


## Generate one sub command

Adjust `SUB_TOPIC`, rerun. Defaults to the device's status topic wildcard so you see every `state_report` it publishes.

In [21]:
SUB_TOPIC = f"{TOPIC_STATUS}/#"

sub_cmd = build_sub_command(SUB_TOPIC)
print(sub_cmd)

mosquitto_sub -h 192.168.1.216 -p 8883 --cafile /Users/nerdmonkey/Productions/microweaver-framework/certs/ca.pem --cert /Users/nerdmonkey/Productions/microweaver-framework/certs/client.pem --key /Users/nerdmonkey/Productions/microweaver-framework/certs/private.pem -u dev_fa6648eb -P $MQTT_PASS -t 'devices/dev_fa6648eb/status/#' -v


Preview the resolved sub command the same way - copies to clipboard, never printed/saved.

In [12]:
preview(sub_cmd)

resolved command copied to clipboard (338 chars, password included)


## Generate a batch

One pub command per enabled actuator's topic suffix, `on`/`off`/`toggle`, plus a `mosquitto_sub` wildcard for status and sensors.

In [28]:
ACTUATOR_KEYS = {
    "relay": ("relay_enabled", "relay_topic_suffix"),
    "rgb": ("rgb_enabled", "rgb_topic_suffix"),
    "oled": ("oled_enabled", "oled_topic_suffix"),
}

commands = []
for name, (enabled_key, suffix_key) in ACTUATOR_KEYS.items():
    if not config.get(enabled_key):
        continue
    suffix = config[suffix_key]
    topic = f"{TOPIC_SUB}/{suffix}"
    for payload in ("on", "off", "toggle"):
        commands.append(build_pub_command(topic, payload))

commands.append(build_sub_command(f"{TOPIC_STATUS}/#"))
commands.append(build_sub_command(f"{TOPIC_PUB}/#"))

for c in commands:
    print(c)

mosquitto_pub -h 192.168.1.216 -p 8883 --cafile /Users/nerdmonkey/Productions/microweaver-framework/certs/ca.pem --cert /Users/nerdmonkey/Productions/microweaver-framework/certs/client.pem --key /Users/nerdmonkey/Productions/microweaver-framework/certs/private.pem -u dev_fa6648eb -P $MQTT_PASS -t devices/dev_fa6648eb/commands/relay -m on
mosquitto_pub -h 192.168.1.216 -p 8883 --cafile /Users/nerdmonkey/Productions/microweaver-framework/certs/ca.pem --cert /Users/nerdmonkey/Productions/microweaver-framework/certs/client.pem --key /Users/nerdmonkey/Productions/microweaver-framework/certs/private.pem -u dev_fa6648eb -P $MQTT_PASS -t devices/dev_fa6648eb/commands/relay -m off
mosquitto_pub -h 192.168.1.216 -p 8883 --cafile /Users/nerdmonkey/Productions/microweaver-framework/certs/ca.pem --cert /Users/nerdmonkey/Productions/microweaver-framework/certs/client.pem --key /Users/nerdmonkey/Productions/microweaver-framework/certs/private.pem -u dev_fa6648eb -P $MQTT_PASS -t devices/dev_fa6648eb/

## Execute a generated command

Runs `mosquitto_pub`/`mosquitto_sub` directly - actually touches the device. `mosquitto_sub` blocks until `timeout` (no `-C`/count limit set), so keep it short for a quick sanity check.

In [29]:
run(build_pub_command(f"{TOPIC_SUB}/relay", "on"));

$ mosquitto_pub -h 192.168.1.216 -p 8883 --cafile /Users/nerdmonkey/Productions/microweaver-framework/certs/ca.pem --cert /Users/nerdmonkey/Productions/microweaver-framework/certs/client.pem --key /Users/nerdmonkey/Productions/microweaver-framework/certs/private.pem -u dev_fa6648eb -P $MQTT_PASS -t devices/dev_fa6648eb/commands/relay -m on
